<a href="https://colab.research.google.com/github/Raka7317/set_project_work/blob/main/last1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================================
# RESEARCH-GRADE HYBRID PHISHING URL DETECTION SYSTEM
# OPTIMIZED VERSION — HIGHER ACCURACY
# ==========================================================

print("\n================ STEP 0: SETUP ================")

from google.colab import drive
drive.mount("/content/drive")

import os, math, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    precision_recall_curve, roc_curve, auc
)
from scipy.sparse import hstack

sns.set(style="whitegrid")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ==========================================================
print("\n================ STEP 1: DATA LOADING ================")

BASE = "/content/drive/MyDrive/data"

def load_data(path):
    df = pd.read_csv(path)
    if "URL" in df.columns:
        df = df.rename(columns={"URL":"url","label":"labels"})
    df = df[["url","labels"]].dropna()
    df["labels"] = df["labels"].astype(int)
    return df

train_df = load_data(os.path.join(BASE,"train.csv"))
test_df  = load_data(os.path.join(BASE,"test.csv"))
phi_df   = load_data(os.path.join(BASE,"phiusiil.csv"))

# PROPER SPLIT: first half trains meta-learner, second half is clean evaluation
# This fixes the label leakage problem in the original code
mid     = len(test_df) // 2
val_df  = test_df.iloc[:mid].reset_index(drop=True)
eval_df = test_df.iloc[mid:].reset_index(drop=True)

print("Train:", train_df.shape)
print("Val  :", val_df.shape,  "<-- meta-learner training set")
print("Eval :", eval_df.shape, "<-- final clean evaluation")
print("PHI  :", phi_df.shape)

print("\nClass Distribution (Train)")
print(train_df.labels.value_counts(normalize=True))

# ==========================================================
print("\n================ STEP 2: LEXICAL FEATURES (EXPANDED 8 -> 15) ================")

BRAND_KEYWORDS = [
    'secure', 'login', 'verify', 'account', 'update',
    'bank', 'paypal', 'amazon', 'apple', 'microsoft'
]

def lexical_features(url):
    u = str(url).lower()
    length        = len(u)
    digit_count   = sum(c.isdigit() for c in u)
    special_count = sum(not c.isalnum() for c in u)
    dot_count     = u.count('.')
    hyphen_count  = u.count('-')
    pct_count     = u.count('%')
    at_count      = u.count('@')
    log_len       = math.log2(length + 1)
    # 7 new features
    slash_count     = u.count('/')
    digit_ratio     = digit_count / (length + 1)
    hyphen_ratio    = hyphen_count / (length + 1)
    has_query       = int('?' in u)
    subdomain_depth = max(0, dot_count - 1)
    has_ip          = int(bool(re.search(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', u)))
    brand_hit       = int(any(kw in u for kw in BRAND_KEYWORDS))
    return [
        length, digit_count, special_count, dot_count, hyphen_count,
        pct_count, at_count, log_len,
        slash_count, digit_ratio, hyphen_ratio, has_query,
        subdomain_depth, has_ip, brand_hit
    ]

X_lex_train = np.array([lexical_features(u) for u in train_df.url])
X_lex_val   = np.array([lexical_features(u) for u in val_df.url])
X_lex_eval  = np.array([lexical_features(u) for u in eval_df.url])
X_lex_phi   = np.array([lexical_features(u) for u in phi_df.url])

scaler_lex  = StandardScaler()
X_lex_train = scaler_lex.fit_transform(X_lex_train)
X_lex_val   = scaler_lex.transform(X_lex_val)
X_lex_eval  = scaler_lex.transform(X_lex_eval)
X_lex_phi   = scaler_lex.transform(X_lex_phi)

print(f"Lexical feature dim: {X_lex_train.shape[1]}  (was 8, now 15)")

# ==========================================================
print("\n================ STEP 3: TF-IDF FEATURES (50K -> 80K vocab) ================")

tfidf = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 6),
    min_df=2,           # was 3
    max_features=80000, # was 50000
    sublinear_tf=True,  # NEW: log-scale TF dampens very frequent n-grams
)

X_ng_train = tfidf.fit_transform(train_df.url)
X_ng_val   = tfidf.transform(val_df.url)
X_ng_eval  = tfidf.transform(eval_df.url)
X_ng_phi   = tfidf.transform(phi_df.url)

print("TF-IDF Feature Dimension:", X_ng_train.shape[1])

X_train_ml = hstack([X_ng_train, X_lex_train]).tocsr()
X_val_ml   = hstack([X_ng_val,   X_lex_val  ]).tocsr()
X_eval_ml  = hstack([X_ng_eval,  X_lex_eval ]).tocsr()
X_phi_ml   = hstack([X_ng_phi,   X_lex_phi  ]).tocsr()

print("Final ML Feature Size:", X_train_ml.shape[1])

# ==========================================================
print("\n================ STEP 4: ONLINE ML MODEL (TUNED) ================")

clf = SGDClassifier(
    loss="log_loss",
    class_weight="balanced",
    alpha=1e-5,      # was default 1e-4, lighter regularization
    max_iter=1000,
    tol=1e-4,
    random_state=42,
    n_jobs=-1,
)
clf.fit(X_train_ml, train_df.labels)

p_ml_val  = clf.predict_proba(X_val_ml)[:,1]
p_ml_eval = clf.predict_proba(X_eval_ml)[:,1]
p_ml_phi  = clf.predict_proba(X_phi_ml)[:,1]

print("ML branch trained.")

# ==========================================================
print("\n================ STEP 5: ISOLATION FOREST (TUNED) ================")

iso = IsolationForest(
    n_estimators=200,    # was default 100
    contamination=0.05,  # was 0.1, tighter boundary
    max_samples='auto',
    random_state=42,
    n_jobs=-1,
)
iso.fit(X_lex_train[train_df.labels.values == 0])

a_val  = -iso.score_samples(X_lex_val)
a_eval = -iso.score_samples(X_lex_eval)
a_phi  = -iso.score_samples(X_lex_phi)

scaler_anom = MinMaxScaler()
a_val  = scaler_anom.fit_transform(a_val.reshape(-1,1)).flatten()
a_eval = scaler_anom.transform(a_eval.reshape(-1,1)).flatten()
a_phi  = scaler_anom.transform(a_phi.reshape(-1,1)).flatten()

print("Isolation Forest trained.")

# ==========================================================
print("\n================ STEP 6: DEEP TCN + ATTENTION (DEEPER) ================")

MAX_LEN = 200  # was 128, captures longer phishing URLs

chars    = sorted(set("".join(train_df.url.astype(str))))
char2idx = {c: i+1 for i, c in enumerate(chars)}
char2idx["<PAD>"] = 0

def encode(u):
    s = [char2idx.get(c, 0) for c in str(u)[:MAX_LEN]]
    return s + [0] * (MAX_LEN - len(s))

class URLDS(Dataset):
    def __init__(self, urls, y):
        self.X = [encode(u) for u in urls]
        self.y = y.values
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return torch.tensor(self.X[i]), torch.tensor(self.y[i], dtype=torch.float)

train_dl = DataLoader(URLDS(train_df.url, train_df.labels), batch_size=512, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(URLDS(val_df.url,   val_df.labels),   batch_size=512, shuffle=False, num_workers=2, pin_memory=True)
eval_dl  = DataLoader(URLDS(eval_df.url,  eval_df.labels),  batch_size=512, shuffle=False, num_workers=2, pin_memory=True)
phi_dl   = DataLoader(URLDS(phi_df.url,   phi_df.labels),   batch_size=512, shuffle=False, num_workers=2, pin_memory=True)

class DeepTCN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.emb     = nn.Embedding(vocab_size, 128, padding_idx=0)  # was 64
        self.conv1   = nn.Conv1d(128, 256, kernel_size=3, padding=1) # was 64->128
        self.conv2   = nn.Conv1d(256, 256, kernel_size=3, padding=1) # was 128->128
        self.conv3   = nn.Conv1d(256, 128, kernel_size=3, padding=1) # NEW 3rd conv
        self.bn      = nn.BatchNorm1d(128)                            # NEW BatchNorm
        self.dropout = nn.Dropout(0.4)                                # was 0.3
        self.attn    = nn.Linear(128, 1)
        self.fc1     = nn.Linear(128, 64)                             # NEW intermediate FC
        self.fc2     = nn.Linear(64, 1)

    def forward(self, x):
        x   = self.emb(x).transpose(1, 2)               # [B, 128, L]
        x   = torch.relu(self.conv1(x))                  # [B, 256, L]
        x   = torch.relu(self.conv2(x))                  # [B, 256, L]
        x   = torch.relu(self.conv3(x))                  # [B, 128, L]
        x   = self.bn(x)                                 # BatchNorm
        x   = self.dropout(x).transpose(1, 2)            # [B, L, 128]
        w   = torch.softmax(self.attn(x).squeeze(-1), dim=1)  # [B, L]
        ctx = (x * w.unsqueeze(-1)).sum(1)               # [B, 128]
        out = torch.relu(self.fc1(ctx))                  # [B, 64]
        return self.fc2(out).squeeze()                   # [B]

model = DeepTCN(len(char2idx)).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

opt   = torch.optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-5)
lossf = nn.BCEWithLogitsLoss()
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=10, eta_min=1e-5)

best_f1    = 0
best_state = None

for e in range(10):  # was 8
    model.train()
    total_loss = 0
    for X, y in train_dl:
        X, y = X.to(device), y.to(device)
        opt.zero_grad()
        loss = lossf(model(X), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # gradient clipping
        opt.step()
        total_loss += loss.item()
    sched.step()

    # Validate and save best checkpoint
    model.eval()
    val_preds, val_true = [], []
    with torch.no_grad():
        for X, y in val_dl:
            probs = torch.sigmoid(model(X.to(device))).cpu().numpy()
            val_preds.extend(probs)
            val_true.extend(y.numpy())
    val_f1 = f1_score(val_true, (np.array(val_preds) > 0.5).astype(int))
    if val_f1 > best_f1:
        best_f1    = val_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    print(f"Epoch {e+1:2d} | Loss: {total_loss/len(train_dl):.4f} | Val F1: {val_f1:.4f}")

model.load_state_dict(best_state)
print(f"\nLoaded best checkpoint — Val F1: {best_f1:.4f}")

def dl_scores(dl):
    model.eval()
    s = []
    with torch.no_grad():
        for X, _ in dl:
            s.extend(torch.sigmoid(model(X.to(device))).cpu().numpy())
    return np.array(s)

p_dl_val  = dl_scores(val_dl)
p_dl_eval = dl_scores(eval_dl)
p_dl_phi  = dl_scores(phi_dl)

# ==========================================================
print("\n================ STEP 7-9: FUSION + META (NO LEAKAGE) ================")

# Meta-learner trained on val_df, evaluated on eval_df
meta_X_val  = np.column_stack([p_ml_val,  p_dl_val,  a_val])
meta_X_eval = np.column_stack([p_ml_eval, p_dl_eval, a_eval])

meta_clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)
meta_clf.fit(meta_X_val, val_df.labels)

print("Meta-learner feature importances [ML, DL, Anomaly]:")
print(np.round(meta_clf.feature_importances_, 4))

S_eval = meta_clf.predict_proba(meta_X_eval)[:,1]
S_phi  = meta_clf.predict_proba(np.column_stack([p_ml_phi, p_dl_phi, a_phi]))[:,1]

# PR-curve optimal threshold
prec, rec, thr = precision_recall_curve(eval_df.labels, S_eval)
f1_arr   = 2 * prec * rec / (prec + rec + 1e-9)
best_thr = thr[np.argmax(f1_arr)]
print(f"\nBest Threshold: {best_thr:.4f}")

yhat_eval = (S_eval > best_thr).astype(int)
yhat_phi  = (S_phi  > best_thr).astype(int)

def print_metrics(y, yhat, s, name):
    print(f"\n{'='*45}")
    print(f"  {name}")
    print(f"{'='*45}")
    print(f"  Accuracy : {accuracy_score(y, yhat):.6f}")
    print(f"  Precision: {precision_score(y, yhat):.6f}")
    print(f"  Recall   : {recall_score(y, yhat):.6f}")
    print(f"  F1 Score : {f1_score(y, yhat):.6f}")
    fpr, tpr, _ = roc_curve(y, s)
    print(f"  AUC      : {auc(fpr, tpr):.6f}")

print_metrics(eval_df.labels, (p_ml_eval > 0.5).astype(int), p_ml_eval, "ML Branch")
print_metrics(eval_df.labels, (p_dl_eval > 0.5).astype(int), p_dl_eval, "DL Branch")
print_metrics(eval_df.labels, yhat_eval, S_eval, "HYBRID — GramBeddings (Final)")
print_metrics(phi_df.labels,  yhat_phi,  S_phi,  "HYBRID — PhiUSIIL (Out-of-Distribution)")

# ==========================================================
print("\n================ STEP 10: DRIFT DETECTION ================")

class PageHinkley:
    def __init__(self, delta=0.002, lam=15):
        self.delta = delta
        self.lam   = lam
        self.reset()
    def reset(self):
        self.mean = 0; self.sum = 0; self.min = 0; self.t = 0
    def update(self, x):
        self.t   += 1
        self.mean += (x - self.mean) / self.t
        self.sum  += x - self.mean - self.delta
        self.min   = min(self.min, self.sum)
        if self.sum - self.min > self.lam:
            self.reset()
            return True
        return False

ph     = PageHinkley()
losses = []
drifts = []

for i in range(len(eval_df)):
    err = int(yhat_eval[i] != eval_df.labels.iloc[i])
    losses.append(err)
    if ph.update(err):
        drifts.append(i)
        print(f"  Drift detected at index {i}")

print(f"\n  Total Drift Points : {len(drifts)}")
print(f"  Drift Rate         : {len(drifts)/len(eval_df):.2e}")

print("\nPIPELINE COMPLETED SUCCESSFULLY")


================ STEP 0: SETUP ================
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda

================ STEP 1: DATA LOADING ================
Train: (160000, 2)
Val  : (80000, 2) <-- meta-learner training set
Eval : (80000, 2) <-- final clean evaluation
PHI  : (235795, 2)

Class Distribution (Train)
labels
1    0.5
0    0.5
Name: proportion, dtype: float64

================ STEP 2: LEXICAL FEATURES (EXPANDED 8 -> 15) ================
Lexical feature dim: 15  (was 8, now 15)

================ STEP 3: TF-IDF FEATURES (50K -> 80K vocab) ================
TF-IDF Feature Dimension: 80000
Final ML Feature Size: 80015

================ STEP 4: ONLINE ML MODEL (TUNED) ================
ML branch trained.

================ STEP 5: ISOLATION FOREST (TUNED) ================
Isolation Forest trained.

================ STEP 6: DEEP TCN + ATTENTION (DEEPER) ================
Model parameters: 414,85